In [ ]:
## Import Dependencies
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import easydict
from tqdm.notebook import tqdm
from sklearn.covariance import LedoitWolf

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
from google.colab import files

## 파일 업로드
uploaded = files.upload()

In [ ]:
## 데이터 불러오기
df = pd.read_csv("chemical_process_timeseries.csv")

## 데이터 확인
df.head()

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    df[df['fault_type']==0]['reactor_temp'],
    bins=50,
    alpha=0.6,
    label='Normal'
)

plt.hist(
    df[df['fault_type']!=0]['reactor_temp'],
    bins=50,
    alpha=0.6,
    label='Fault'
)

plt.legend()

plt.title("Normal vs Fault Sensor Distribution")

plt.show()

### Regime A만 사용

In [ ]:
df = df[df['operating_regime'] == 'A'].copy()

In [ ]:
features = [
    "reactor_temp", "reactor_pressure", "feed_flow_rate", "coolant_flow_rate",
    "agitator_speed_rpm", "vibration_rms", "motor_current", "power_consumption_kw"
]

In [ ]:
df = df[['timestamp', 'reactor_id', 'fault_type'] + features]

In [ ]:
df.head()

### 데이터 전처리

In [ ]:
## 데이터 Type 변경
df['timestamp'] = pd.to_datetime(df['timestamp'])

## Reactor별 시간 순서 정렬
df = df.sort_values(['reactor_id', 'timestamp']).reset_index(drop=True)

In [ ]:
## 결측 변수 확인
(df.isnull().sum()/len(df)).plot.bar(figsize=(18,8), colormap='Paired')

In [ ]:
## 이전 시점의 데이터로 보간
df[features] = (
    df.groupby('reactor_id')[features]
    .transform(lambda x: x.ffill().bfill())
)

## timestamp를 index로 변환
df = df.set_index('timestamp')

### 데이터 분리 및 정규화

In [ ]:
## 리액터별로 고장과 정상 데이터가 몇 개씩 있는지 확인
breakdown = df.groupby(['reactor_id', df['fault_type'] != 0]).size().unstack(fill_value=0)
breakdown.columns = ['Normal (0)', 'Fault (1)']
print(breakdown)

In [ ]:
## 정상 데이터와 비정상 데이터 분리하여 학습, 검증, 테스트
normal_df = df[df['fault_type'] == 0]
abnormal_df = df[df['fault_type'] != 0]

In [ ]:
## normal/abnormal 비율
total = len(df)

normal_ratio = len(normal_df) / total * 100
abnormal_ratio = len(abnormal_df) / total * 100

print(f"Normal Ratio: {normal_ratio:.2f}%")
print(f"Abnormal Ratio: {abnormal_ratio: .2f}%")

In [ ]:
## fault tpye별 비율
fault_summary = pd.DataFrame({
    'Count': df['fault_type'].value_counts(),
    'Ratio (%)': df['fault_type'].value_counts(normalize=True) * 100
})

print(fault_summary.sort_index())

In [ ]:
## reactor별 시간 순서 유지한 상태로 정상 데이터 split
train_list = []
val_list = []
test_list = []

## reactor별 반복
for reactor_id, reactor_df in normal_df.groupby('reactor_id'):
  reactor_df = reactor_df.sort_index()

  ## 데이터 길이
  n = len(reactor_df)

  ## 70/15/15 split
  train_end = int(n * 0.7)
  val_end = int(n * 0.85)

  ## 시간 기준 split
  train_part = reactor_df.iloc[:train_end]
  val_part = reactor_df.iloc[train_end:val_end]
  test_part = reactor_df.iloc[val_end:]

  ## 리스트에 저장
  train_list.append(train_part)
  val_list.append(val_part)
  test_list.append(test_part)

## reactor별 분할된 데이터 합치기
train_normal = pd.concat(train_list)
val_normal = pd.concat(val_list)
test_normal = pd.concat(test_list)

## abnormal 데이터는 전체를 테스트에 사용
test_abnormal = abnormal_df.copy()

In [ ]:
## 데이터 정규화를 위하여 분산 및 평균 추출
mean_df = train_normal[features].mean()
std_df = train_normal[features].std()

### 데이터 구조 만들기

In [ ]:
## 데이터를 불러올 때 index로 불러오기
def make_data_idx(df, window_size=60, stride=10):

  input_idx = []

  df_reset = df.reset_index()

  ## reactor별로 그룹화하여 데이터가 섞이지 않게 함
  for reactor_id, group in df_reset.groupby('reactor_id'):

    group = group.sort_values('timestamp') # 시간 정렬

    indices = group.index.tolist()
    timestamps = group['timestamp'].values # timestamps 추출

		## sliding window 생성
    for i in range(
        0,
        len(group) - window_size + 1,
        stride
    ):

      start_time = timestamps[i]
      end_time = timestamps[i + window_size - 1]

      ## 정확히 1분 간격인지 확인
      diff = (
          pd.Timestamp(end_time) - pd.Timestamp(start_time)
      ).total_seconds() / 60

      if diff == (window_size - 1):

        seq_idx = indices[i:i+window_size]
        input_idx.append(seq_idx)

  return input_idx

In [ ]:
## Dataset을 상속받아 데이터 구성
class ChemicalDataset(Dataset):
  def __init__(self, input_size, df, mean_df=None, std_df=None, window_size=60, stride=10):

    self.input_size = input_size
    self.window_size = window_size

    temp_df = df.copy()

    ## 정규화
    if mean_df is not None:
      temp_df[features] = (temp_df[features] - mean_df) / std_df

    ## 연속한 index를 기준으로 학습에 사용
    self.input_ids = make_data_idx(temp_df, window_size=window_size, stride=stride)

    ## sensor 데이터만 사용하여 reconstruct에 활용
    self.features = features[:input_size]
    self.var_data = torch.tensor(temp_df[self.features].values, dtype=torch.float)

    ## metadate 저장
    self.meta_df = temp_df.reset_index()

  ## Dataset은 반드시 __len__ 함수를 만들어줘야함
  ## sequence 개수 반환
  def __len__(self):
    return len(self.input_ids)

  ## Dataset은 반드시 __getitem__ 함수를 만들어줘야함
  ## torch 모듈은 __getitem__ 을 호출하여 학습할 데이터를 불러옴
  def __getitem__(self, idx):
    seq_idx = self.input_ids[idx]
    x = self.var_data[seq_idx]
    return x

In [ ]:
def get_sequence_fault_labels(dataset):

    labels = []

    for seq_idx in dataset.input_ids:

        ## sequence 마지막 시점 기준
        last_idx = seq_idx[-1]
        fault_label = dataset.meta_df.iloc[last_idx]['fault_type']
        labels.append(fault_label)

    return np.array(labels)

### 모델 구성하기

In [ ]:
## 인코더
class Encoder(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers=2):

    super().__init__()

    self.lstm = nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True,
        dropout=0.1
    )

  def forward(self, x):
    _, (hidden, cell) = self.lstm(x)

    ## 마지막 layer latent
    latent = hidden[-1]

    return hidden, cell, latent

In [ ]:
## 디코더
class Decoder(nn.Module):
  def __init__(self, input_size, hidden_size, output_size, num_layers=2):

    super(Decoder, self).__init__()

    self.lstm = nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True,
        dropout=0.1
    )

    self.fc = nn.Linear(hidden_size, output_size)

  def forward(self, x, hidden):
    output, (hidden, cell) = self.lstm(x, hidden)
    prediction = self.fc(output)
    return prediction, (hidden, cell)

In [ ]:
## LSTM Auto Encoder
class LSTMAutoEncoder(nn.Module):
  def __init__(self, input_dim, latent_dim, window_size=60, num_layers=2):
    super().__init__()

    self.encoder = Encoder(input_dim, latent_dim, num_layers)
    self.decoder = Decoder(input_size=input_dim, hidden_size=latent_dim, output_size=input_dim, num_layers=num_layers)
    self.window_size = window_size

  def forward(self, src: torch.Tensor):
    batch_size, sequence_length, var_length = src.size()

    ## Encoder
    hidden, cell, latent = self.encoder(src)
    encoder_states = (hidden, cell)

    inv_idx = torch.arange(sequence_length -1, -1, -1).long()

    reconstruct_output = []

		## temp_input 초기화
    temp_input = torch.zeros(
        (batch_size, 1, var_length),
        dtype=torch.float,
    ).to(src.device)

		## Encoder memory를 Decoder 시작 상태로 사용
    decoder_hidden = encoder_states

		## sequence 복원 (batch, 1, feature)
    for t in range(sequence_length):
      temp_input, decoder_hidden = self.decoder(
          temp_input,
          decoder_hidden
      )

      reconstruct_output.append(temp_input)

		## sequence 합치기 (batch, sequence_length, feature)
		## Decoder reconstruction 순서를 다시 원래 시간순으로 맞추기 위해 reverse로 순서 복구
    reconstruct_output = torch.cat(reconstruct_output, dim=1)[:, inv_idx, :]

    return reconstruct_output, src

  def loss_function(self, recons, target):

    ## MSE loss(Mean squared Error) → 원본과 얼마나 다른지 측정
    loss = F.mse_loss(recons, target)
    return loss

### 학습 구성

In [ ]:
def run(args, model, train_loader, val_loader):

  ## optimizer 설정
  optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate)

  best_loss = np.inf
  patience_count = 0

  epochs = tqdm(range(args.epochs))
  for epoch in epochs:

    ## TRAIN
    model.train()

    train_loss = 0
    for batch_data in train_loader:
      batch_data = batch_data.to(args.device)
      recons, target = model(batch_data)
      loss = model.loss_function(recons, target)

      ## Backward and optimize
      optimizer.zero_grad()
      loss.backward()

      torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
      )

      optimizer.step()
      train_loss += loss.item()

    ## VALIDATION
    model.eval()

    val_loss = 0

    with torch.no_grad():

      for batch_data in val_loader:
        batch_data = batch_data.to(args.device)
        recons, target = model(batch_data)

        loss = model.loss_function(recons, target)
        val_loss += loss.item()

    ## 평균 loss
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    epochs.set_postfix({
        "Train Loss": train_loss,
        "Validation Loss": val_loss
    })

    ## Early stopping
    if val_loss < best_loss:
      best_loss = val_loss
      patience_count = 0
    else:
      patience_count += 1

      if patience_count >= args.patience:
        print("Early Stopping")
        break

  return model

def get_reconstruction_errors(args, model, data_loader):

  model.eval()
  errors = []

  with torch.no_grad():
    for batch_data in data_loader:

      batch_data = batch_data.to(args.device)
      recons, target = model(batch_data)

      ## feature별 MAE(Mean Absolute Error)
      loss = torch.abs(recons - target)

      ## sequence 평균
      loss = loss.mean(dim=1)
      errors.append(loss.cpu().numpy())

  errors = np.concatenate(errors, axis=0)

  return errors

### 모델 & 학습파라미터 설정

In [ ]:
## 하이퍼 파라미터 설정
args = easydict.EasyDict({
    "batch_size": 128, # batch
    "device": torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'), # device
    "input_size": len(features), # feature dimension
    "latent_size": 16, # latent dimension
    "window_size": 60, # sequence length
    "num_layers": 2, # LSTM depth
    "learning_rate": 1e-3, # optimizer
    "epochs": 40, # epochs
    "early_stop": True, # early stopping
    "patience": 5, # patience
    "stride": 10 # sliding window stride
})

### 학습하기

In [ ]:
## 데이터셋 생성 (train, validation, abnormal)
train_dataset = ChemicalDataset(args.input_size, train_normal, mean_df, std_df, args.window_size, stride=args.stride)
val_dataset = ChemicalDataset(args.input_size, val_normal, mean_df, std_df, args.window_size, stride=args.stride)
test_normal_dataset = ChemicalDataset(args.input_size, test_normal, mean_df, std_df, args.window_size, stride=args.stride)
test_abnormal_dataset = ChemicalDataset(args.input_size, test_abnormal, mean_df, std_df, args.window_size, stride=args.stride)

## Data Loader 형태로 변환
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
test_normal_loader = DataLoader(test_normal_dataset, batch_size=args.batch_size, shuffle=False)
test_abnormal_loader = DataLoader(test_abnormal_dataset, batch_size=args.batch_size, shuffle=False)

In [ ]:
## 모델 생성
model = LSTMAutoEncoder(
    input_dim=args.input_size,
    latent_dim=args.latent_size,
    window_size=args.window_size,
    num_layers=args.num_layers
    ).to(args.device)

## 학습하기
model = run(args, model, train_loader, val_loader)

### 마할라노비스 거리

In [ ]:
## validation 정상 데이터의 reconstruction error vector
normal_errors = get_reconstruction_errors(args, model, val_loader)

## 일반 공분산은 feature 간 상관관계가 많으면 불안정해지므로 LedoitWolf 통해 안정화
lw = LedoitWolf()
lw.fit(normal_errors)

err_mean = lw.location_
inv_cov = lw.precision_

## 마할라노비스 거리 계산
def calc_mahalanobis(err_vec):
  diff = err_vec - err_mean

  # (x-μ)^T * Σ^-1 * (x-μ) 의 제곱근
  md = np.sqrt(
      np.maximum(
          np.dot(np.dot(diff, inv_cov), diff.T),
          0
      )
  )

  return md

## 정상 validation sequence 각각의 anomaly score 계산
normal_md_scores = np.array([
    calc_mahalanobis(e)
    for e in normal_errors
])

## Threshold 설정
threshold = np.percentile(normal_md_scores, 99)
print(f"Threshold: {threshold:.4f}")

## Test Normal Reconstruction Erros
test_normal_errors = get_reconstruction_errors(args, model, test_normal_loader)

## Mahalanobis Scores
test_normal_md_scores = np.array([
    calc_mahalanobis(e)
    for e in test_normal_errors
    ])

## Test Abnormal Reconstruction Errors
test_abnormal_errors = get_reconstruction_errors(args, model, test_abnormal_loader)

## Mahalanobis Scores
test_abnormal_md_scores = np.array([
    calc_mahalanobis(e)
    for e in test_abnormal_errors
    ])

## 실제 이상인데 MD 점수가 임계치보다 낮으면 미탐(False Negative)
missed_detections = np.sum(test_abnormal_md_scores < threshold)
p_monitor = missed_detections / len(test_abnormal_md_scores)

print(f"Total Abnormal Sequences: {len(test_abnormal_md_scores)}")
print(f"Missed Detections: {missed_detections}")
print(f"P_monitor: {p_monitor * 100:.2f}%")

### 시각화

In [ ]:
from sklearn.metrics import confusion_matrix

## Confusion Matrix
normal_pred = (test_normal_md_scores > threshold).astype(int)
abnormal_pred = (test_abnormal_md_scores > threshold).astype(int)

normal_true = np.zeros(len(normal_pred))
abnormal_true = np.ones(len(abnormal_pred))

y_true = np.concatenate([normal_true, abnormal_true])
y_pred = np.concatenate([normal_pred, abnormal_pred])

cm = confusion_matrix(y_true, y_pred)

TN, FP, FN, TP = cm.ravel()

print("Confusion Matrix")
print(cm)

## plot
plt.figure(figsize=(5,4))
plt.imshow(cm)

plt.xticks([0, 1], ['Normal', 'Abnormal'])
plt.yticks([0, 1], ['Normal', 'Abnormal'])

plt.xlabel('Predicted')
plt.ylabel('True')

plt.title('Confusion Matrix')

for i in range(2):
  for j in range(2):
    plt.text(j, i, str(cm[i, j]), ha='center', va='center')

plt.colorbar()
plt.show()

In [ ]:
## MD Score Distribution
print("Normal MD")
print(normal_md_scores.min(), normal_md_scores.max())

print("Abnormal MD")
print(test_abnormal_md_scores.min(), test_abnormal_md_scores.max())

## Log-scale bins 생성
min_score = min(normal_md_scores.min(), test_abnormal_md_scores.min())
if min_score <= 0:
  min_score = 0.1

max_score = max(normal_md_scores.max(), test_abnormal_md_scores.max())

log_bins = np.logspace(np.log10(min_score), np.log10(max_score), 100)

## Plot
plt.figure(figsize=(10, 6))

plt.hist(normal_md_scores, bins=log_bins, alpha=0.6, label='Normal', density=True)
plt.hist(test_abnormal_md_scores, bins=log_bins, alpha=0.6, label='Abnormal', density=True)

## Threshold
plt.axvline(
    threshold,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f'Threshold={threshold:.2f}'
)

## x축 제한
plt.xscale('log')

plt.xlabel("Mahalanobis Distance")
plt.ylabel("Density")
plt.title("Anomaly Score Distribution")

plt.legend()
plt.show()

In [ ]:
## fault_type별 미탐률
abnormal_fault_labels = get_sequence_fault_labels(test_abnormal_dataset)

fault_results =[]

for fault_id in sorted(np.unique(abnormal_fault_labels)):

  ## 정상 제외
  if fault_id == 0:
    continue

  ## 해당 fault만 추출
  idx = abnormal_fault_labels == fault_id
  fault_scores = test_abnormal_md_scores[idx]

  ## 미탐
  missed = np.sum(fault_scores < threshold)
  total = len(fault_scores)
  miss_rate = missed / total

  fault_results.append([fault_id, total, missed, miss_rate])

  print(f"\nFault Type {fault_id}")
  print(f"Total Sequences: {total}")
  print(f"Missed Detections: {missed}")
  print(f"Miss Rate: {miss_rate * 100:.2f}%")

## plot
fault_df = pd.DataFrame(
    fault_results,
    columns=["Fault Type", "Total Sequences", "Missed Detections", "Miss Rate"]
)

plt.figure(figsize=(10,6))
plt.bar(fault_df["Fault Type"].astype(str), fault_df["Miss Rate"])
plt.xlabel("Fault Type")
plt.ylabel("Missed Detection Rate")
plt.title("Missed Detection Rate by Fault Type")
plt.show()

In [ ]:
## Threshold Sensitivity Analysis
percentiles = [90, 92, 95, 97, 98, 99, 99.5, 99.9]

threshold_results = []

for p in percentiles:

    ## Threshold 계산
    threshold = np.percentile(normal_md_scores, p)

    ## Prediction
    normal_pred = (test_normal_md_scores > threshold).astype(int)
    abnormal_pred = (test_abnormal_md_scores > threshold).astype(int)

    ## True label
    normal_true = np.zeros(len(normal_pred))
    abnormal_true = np.ones(len(abnormal_pred))

    ## Concatenate
    y_true = np.concatenate([normal_true, abnormal_true])
    y_pred = np.concatenate([normal_pred, abnormal_pred])

    ## Confusion Matrix
    cm = confusion_matrix(
        y_true,
        y_pred
    )

    TN, FP, FN, TP = cm.ravel()

    ## Metric 계산
    miss_rate = FN / (FN + TP)
    false_alarm_rate = FP / (FP + TN)
    recall = TP / (TP + FN)
    precision = TP / (TP + FP + 1e-8)

    threshold_results.append([p, threshold, miss_rate, false_alarm_rate, precision, recall])

results_df = pd.DataFrame(
  threshold_results,
  columns=[
    "Percentile",
    "Threshold",
    "Miss Rate",
    "False Alarm Rate",
    "Precision",
    "Recall"
  ]
)

print(results_df)

## Plot
plt.figure(figsize=(6,4))

plt.plot(
    results_df["Threshold"],
    results_df["Miss Rate"],
    marker='o'
)

plt.xlabel("Threshold")
plt.ylabel("Miss Rate")
plt.title("Threshold vs Miss Rate")

plt.grid(True)

plt.show()

plt.figure(figsize=(6,4))

plt.plot(
    results_df["Threshold"],
    results_df["False Alarm Rate"],
    marker='o'
)

plt.xlabel("Threshold")
plt.ylabel("False Alarm Rate")
plt.title("Threshold vs False Alarm Rate")

plt.grid(True)

plt.show()

plt.figure(figsize=(6,6))

plt.plot(
    results_df["False Alarm Rate"],
    results_df["Miss Rate"],
    marker='o'
)

for i in range(len(results_df)):

    plt.text(
        results_df["False Alarm Rate"][i],
        results_df["Miss Rate"][i],
        str(results_df["Percentile"][i])
    )

plt.xlabel("False Alarm Rate")
plt.ylabel("Miss Rate")

plt.title("Risk Trade-off Curve")

plt.grid(True)

plt.show()